# NEPR — Network Entanglement Pair Reservation

Este notebook demonstra o protocolo **NEPR** na camada de aplicação do QuantumNet.

- Topologia: grade **3×4** (12 nós)
- Dois nós sorteados aleatoriamente como Alice e Bob
- **10 requisições** NEPR entre os mesmos nós
- Log de execução gravado em `nepr.log`
- Métricas de eventos gravadas em `nepr_metrics.csv`

## 1. Imports

In [11]:
import sys
sys.path.append('../') 

import pandas as pd
import random
import json

from quantumnet.topology import Network
from quantumnet.runtime import Clock
from quantumnet.config import SimulationConfig
from quantumnet.utils import MetricsCollector, Logger

## 2. Parâmetros da simulação

In [12]:
ROWS         = 3          # linhas da grade
COLS         = 4          # colunas da grade  →  12 nós no total
NUM_REQUESTS = 10         # número de requisições NEPR
NUM_PAIRS    = 3          # pares EPR por requisição
SEED         = 1         # semente para reprodutibilidade

LOG_FILE     = 'nepr.log'
METRICS_CSV  = 'nepr_metrics.csv'

print(f"Topologia : Grade {ROWS}×{COLS} = {ROWS * COLS} nós")
print(f"Requisições : {NUM_REQUESTS} × {NUM_PAIRS} pares EPR")
print(f"Log        → {LOG_FILE}")
print(f"Métricas   → {METRICS_CSV}")

Topologia : Grade 3×4 = 12 nós
Requisições : 10 × 3 pares EPR
Log        → nepr.log
Métricas   → nepr_metrics.csv


## 3. Ativar logger externo

In [13]:
logger = Logger.get_instance()
logger.activate(level='DEBUG', filename=LOG_FILE, console=False, file_log=True)

print(f"Logger ativado — saída em '{LOG_FILE}'")

Logger ativado — saída em 'nepr.log'


## 4. Configuração da rede

In [14]:
config = SimulationConfig()
config.defaults.qubits_per_host      = 10   # qubits iniciais por nó
config.defaults.qubit_regen_interval = 10   # ticks entre ciclos de regeneração
config.defaults.qubit_regen_amount   = 5    # qubits adicionados por nó por ciclo

clock = Clock()
net   = Network(clock=clock, config=config)

print("Rede instanciada com Clock e SimulationConfig.")
print(f"Qubits iniciais por nó     : {config.defaults.qubits_per_host}")
print(f"Intervalo de regeneração   : {config.defaults.qubit_regen_interval} ticks")
print(f"Qubits por ciclo (por nó)  : {config.defaults.qubit_regen_amount}")

Rede instanciada com Clock e SimulationConfig.
Qubits iniciais por nó     : 10
Intervalo de regeneração   : 10 ticks
Qubits por ciclo (por nó)  : 5


## 5. Definir topologia e sortear nós

In [15]:
random.seed(SEED)
all_nodes = list(range(ROWS * COLS))
alice_id, bob_id = random.sample(all_nodes, 2)

print(f"Nós disponíveis : {all_nodes}")
print(f"Alice (origem)  : nó {alice_id}")
print(f"Bob   (destino) : nó {bob_id}")
print()
print("Layout da grade 3×4:")
for r in range(ROWS):
    row_nodes = [r * COLS + c for c in range(COLS)]
    markers   = [
        f"[A={n}]" if n == alice_id else
        f"[B={n}]" if n == bob_id  else
        f"  {n:2}  "
        for n in row_nodes
    ]
    print("  ".join(markers))

Nós disponíveis : [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]
Alice (origem)  : nó 2
Bob   (destino) : nó 9

Layout da grade 3×4:
   0       1    [A=2]     3  
   4       5       6       7  
   8    [B=9]    10      11  


## 6. Executar 10 requisições NEPR

In [16]:
results = []

def make_callback(req_id):
    """Retorna um callback que captura o resultado e dispara a próxima requisição."""
    def on_complete(success, measurements):
        meas  = measurements or []
        avg_f = round(sum(meas) / len(meas), 6) if meas else None
        results.append({
            'requisição'       : req_id + 1,
            'alice'            : alice_id,
            'bob'              : bob_id,
            'pares_solicitados': NUM_PAIRS,
            'pares_medidos'    : len(meas),
            'sucesso'          : success,
            'fidelidade_média' : avg_f,
            'medições'         : meas,
        })
        # Encadeia a próxima requisição (serialização)
        next_id = req_id + 1
        if next_id < NUM_REQUESTS:
            net.application_layer.nepr_protocol(
                alice_id, bob_id, NUM_PAIRS,
                on_complete=make_callback(next_id)
            )
        else:
            # Última requisição concluída — para a regeneração
            net.physical.stop_qubit_regen()
    return on_complete


with MetricsCollector(clock, METRICS_CSV) as col:
    # A topologia é definida DENTRO do contexto para capturar eventos de setup
    net.set_ready_topology('Grid', ROWS, COLS)

    # Dispara apenas a primeira requisição; as demais são encadeadas via callback
    net.application_layer.nepr_protocol(
        alice_id, bob_id, NUM_PAIRS,
        on_complete=make_callback(0)
    )

    # Executa a simulação — termina quando não há mais eventos pendentes
    clock.run()

print(f"Simulação concluída — {len(results)} callbacks recebidos.")
print(f"Métricas salvas em '{METRICS_CSV}'.")

Simulação concluída — 10 callbacks recebidos.
Métricas salvas em 'nepr_metrics.csv'.


## 7. Resultados das medições

In [17]:
df_results = pd.DataFrame(results)

# Tabela resumida (sem a coluna de lista de medições)
print("=== Resumo das 10 requisições NEPR ===")
display(df_results.drop(columns=['medições']))

# Estatísticas gerais
sucessos   = df_results['sucesso'].sum()
falhas     = NUM_REQUESTS - sucessos
fids_valid = df_results.loc[df_results['sucesso'], 'fidelidade_média'].dropna()

print()
print(f"Sucesso : {sucessos}/{NUM_REQUESTS}")
print(f"Falha   : {falhas}/{NUM_REQUESTS}")
if not fids_valid.empty:
    print(f"Fidelidade média (sucessos) : {fids_valid.mean():.6f}")
    print(f"Fidelidade mínima          : {fids_valid.min():.6f}")
    print(f"Fidelidade máxima          : {fids_valid.max():.6f}")

=== Resumo das 10 requisições NEPR ===


,requisição,alice,bob,pares_solicitados,pares_medidos,sucesso,fidelidade_média
0,1,2,9,3,0,False,NaN
1,2,2,9,3,3,True,0.581630
2,3,2,9,3,3,True,0.498747
3,4,2,9,3,3,True,0.498747
4,5,2,9,3,3,True,0.498747
5,6,2,9,3,3,True,0.498747
6,7,2,9,3,3,True,0.498747
7,8,2,9,3,3,True,0.498747
8,9,2,9,3,3,True,0.498747
9,10,2,9,3,3,True,0.498747



Sucesso : 9/10
Falha   : 1/10
Fidelidade média (sucessos) : 0.507956
Fidelidade mínima          : 0.498747
Fidelidade máxima          : 0.581630


## 8. Medições individuais por requisição

In [18]:
print("=== Fidelidades individuais por requisição ===")
for row in results:
    status = "OK" if row['sucesso'] else "FALHOU"
    meas_str = ", ".join(f"{f:.6f}" for f in row['medições']) if row['medições'] else "—"
    print(f"  Req {row['requisição']:>2} [{status}]  "
          f"Alice={row['alice']} → Bob={row['bob']}  "
          f"fidelidades=[{meas_str}]  "
          f"média={row['fidelidade_média']}")

=== Fidelidades individuais por requisição ===
  Req  1 [FALHOU]  Alice=2 → Bob=9  fidelidades=[—]  média=None
  Req  2 [OK]  Alice=2 → Bob=9  fidelidades=[0.205891, 0.810000, 0.729000]  média=0.58163
  Req  3 [OK]  Alice=2 → Bob=9  fidelidades=[0.313811, 0.282430, 0.900000]  média=0.498747
  Req  4 [OK]  Alice=2 → Bob=9  fidelidades=[0.313811, 0.282430, 0.900000]  média=0.498747
  Req  5 [OK]  Alice=2 → Bob=9  fidelidades=[0.313811, 0.282430, 0.900000]  média=0.498747
  Req  6 [OK]  Alice=2 → Bob=9  fidelidades=[0.313811, 0.282430, 0.900000]  média=0.498747
  Req  7 [OK]  Alice=2 → Bob=9  fidelidades=[0.313811, 0.282430, 0.900000]  média=0.498747
  Req  8 [OK]  Alice=2 → Bob=9  fidelidades=[0.313811, 0.282430, 0.900000]  média=0.498747
  Req  9 [OK]  Alice=2 → Bob=9  fidelidades=[0.313811, 0.282430, 0.900000]  média=0.498747
  Req 10 [OK]  Alice=2 → Bob=9  fidelidades=[0.313811, 0.282430, 0.900000]  média=0.498747


## 9. Eventos registrados no CSV de métricas

In [19]:
df_metrics = pd.read_csv(METRICS_CSV)

print(f"Total de eventos registrados: {len(df_metrics)}")
print(f"Tipos de eventos: {df_metrics['event_type'].unique().tolist()}")
print()

# Filtrar apenas eventos da camada de aplicação (NEPR)
nepr_mask    = df_metrics['event_type'].isin(['nepr_complete', 'nepr_failed'])
df_nepr_evts = df_metrics[nepr_mask].reset_index(drop=True)

print(f"=== Eventos NEPR ({len(df_nepr_evts)}) ===")
display(df_nepr_evts)

Total de eventos registrados: 3115
Tipos de eventos: ['qubit_created', 'epr_created', 'link_request_attempt', 'echp_success', 'link_request_success', 'echp_low_fidelity', 'qubits_regenerated', 'route_lookup', 'route_found', 'epr_request_failed', 'nepr_failed', 'epr_expired', 'qubit_expired', 'entanglement_swapping_complete', 'epr_request_complete', 'nepr_complete']

=== Eventos NEPR (10) ===


,clock_tick,event_type,source_node,target_node,value,details
0,13,nepr_failed,2.0,9.0,3.0,"{""reason"":""entanglement_failed"",""created"":0}"
1,32,nepr_complete,2.0,9.0,NaN,"{""num_pairs"":3,""measurements"":""[0.205891132094..."
2,51,nepr_complete,2.0,9.0,NaN,"{""num_pairs"":3,""measurements"":""[0.313810596090..."
3,71,nepr_complete,2.0,9.0,NaN,"{""num_pairs"":3,""measurements"":""[0.313810596090..."
4,91,nepr_complete,2.0,9.0,NaN,"{""num_pairs"":3,""measurements"":""[0.313810596090..."
5,111,nepr_complete,2.0,9.0,NaN,"{""num_pairs"":3,""measurements"":""[0.313810596090..."
6,131,nepr_complete,2.0,9.0,NaN,"{""num_pairs"":3,""measurements"":""[0.313810596090..."
7,151,nepr_complete,2.0,9.0,NaN,"{""num_pairs"":3,""measurements"":""[0.313810596090..."
8,171,nepr_complete,2.0,9.0,NaN,"{""num_pairs"":3,""measurements"":""[0.313810596090..."
9,191,nepr_complete,2.0,9.0,NaN,"{""num_pairs"":3,""measurements"":""[0.313810596090..."


## 10. Inspecionar o arquivo de log externo

In [20]:
import os

if os.path.exists(LOG_FILE):
    size_kb = os.path.getsize(LOG_FILE) / 1024
    with open(LOG_FILE, 'r', encoding='utf-8') as f:
        lines = f.readlines()
    print(f"Arquivo '{LOG_FILE}' — {len(lines)} linhas / {size_kb:.1f} KB")
    print()
    print("--- Últimas 30 linhas do log ---")
    print(''.join(lines[-30:]))
else:
    print(f"Arquivo '{LOG_FILE}' não encontrado (o logger pode estar desativado).")

UnicodeDecodeError: 'utf-8' codec can't decode byte 0x97 in position 1252: invalid start byte